# Support Vector Machines — Implementations

The linear SVM twice more: once with autograd differentiating the very same mini-batch hinge objective (identical batches, identical branch at the kink, so the runs agree to machine zero), and once through sklearn, whose exact solver shows how far a fixed-step subgradient loop hovers from the true minimiser — which is why that lane compares predictions, not coefficients.

## 09_linear_svm

Maximum margin by subgradient descent on the hinge loss.

### torch

The scratch loop with autograd replacing the hand-written gradient: the violator mask is taken from *detached* margins with the scratch's strict `margins < 1`, the hinge is averaged over violators only, and NumPy still draws the permutations — so both lanes see the same batches and agree bit for bit. **What torch adds:** the update now comes from the objective, not from a derivation on paper — change the loss and the gradient follows for free, which the checks confirm term by term.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Build the violator mask from detached margins with the scratch's strict `< 1`.
# 2. Average hinge over violators only; relu(1-m).mean() divides by the wrong count.
# 3. Keep the shuffle in NumPy: same random_state, same permutations, same batches.
# 4. A violation-free batch leaves b.grad None — the scratch's grad_b = 0, in torch.
# 5. float64 end to end; zero init and fixed epochs make the lanes bit-comparable.


class LinearSVCScratch:
    """Linear soft-margin SVM: the scratch's sub-gradient loop with autograd
    doing the differentiation.

    Minimizes: (1/2)||w||^2 + C * mean(max(0, 1 - y_i * (w @ x_i + b)))
    """

    def __init__(self, C=1.0, lr=0.01, max_iter=1000, batch_size=32, random_state=42):
        self.C = C
        self.lr = lr
        self.max_iter = max_iter
        self.batch_size = batch_size
        self.random_state = random_state
        self.w_ = None
        self.b_ = None
        self.classes_ = None
        self.loss_history_ = []

    def _encode_labels(self, y):
        """Map original labels to (-1, 1)."""
        self.classes_ = np.unique(y)
        if len(self.classes_) != 2:
            raise ValueError('LinearSVCScratch supports exactly two classes')
        return np.where(y == self.classes_[0], -1.0, 1.0)

    def _hinge_loss(self, X, y_enc):
        """Compute the full objective: 0.5*||w||^2 + C*mean(hinge)."""
        margins = y_enc * (X @ self.w_ + self.b_)
        hinge = np.maximum(0, 1 - margins)
        return 0.5 * np.dot(self.w_, self.w_) + self.C * np.mean(hinge)

    def fit(self, X, y):
        """Same batches, same mask, but loss.backward() produces the update."""
        X = np.asarray(X, dtype=float)
        y_enc = self._encode_labels(y)
        n, p = X.shape

        Xt = torch.as_tensor(X)
        yt = torch.as_tensor(y_enc)
        w = torch.zeros(p, dtype=torch.float64, requires_grad=True)
        b = torch.zeros((), dtype=torch.float64, requires_grad=True)
        self.loss_history_ = []

        rng_fit = np.random.default_rng(self.random_state)
        batch_size = min(self.batch_size, n)

        for epoch in range(self.max_iter):
            indices = rng_fit.permutation(n)
            for start in range(0, n, batch_size):
                batch_idx = indices[start:start + batch_size]
                margins = yt[batch_idx] * (Xt[batch_idx] @ w + b)

                # The scratch's strict `margins < 1`, taken on detached values:
                # a point at margin exactly 1 contributes nothing in either lane.
                violations = margins.detach() < 1.0

                # Hinge averaged over violators only — the scratch's convention.
                # `relu(1 - margins).mean()` over the batch would divide by the
                # batch size instead and the lanes would drift apart.
                loss = 0.5 * (w @ w)
                if bool(violations.any()):
                    loss = loss + self.C * (1.0 - margins[violations]).mean()

                w.grad = None
                b.grad = None
                loss.backward()
                with torch.no_grad():
                    w -= self.lr * w.grad
                    if b.grad is not None:  # no violators: the scratch's grad_b = 0
                        b -= self.lr * b.grad

            self.w_ = w.detach().numpy().copy()
            self.b_ = float(b.detach())
            self.loss_history_.append(self._hinge_loss(X, y_enc))

        return self

    def decision_function(self, X):
        """Compute w^T x + b for each sample."""
        return np.asarray(X, dtype=float) @ self.w_ + self.b_

    def predict(self, X):
        """Predict original class labels."""
        raw = (self.decision_function(X) >= 0.0).astype(int)
        return self.classes_[raw]

    def score(self, X, y):
        """Classification accuracy."""
        return np.mean(self.predict(X) == y)

    def support_vectors(self, X, y):
        """Identify support vectors: points with functional margin <= 1."""
        y_enc = np.where(y == self.classes_[0], -1.0, 1.0)
        margins = y_enc * (X @ self.w_ + self.b_)
        sv_mask = margins <= 1.0 + 1e-3  # small tolerance
        return X[sv_mask], np.where(sv_mask)[0]


In [ ]:
# exports: w_eq, b_eq, loss_final, pred_test
_rng_eq = np.random.default_rng(909)
X_eq = np.vstack([_rng_eq.normal(loc=[2.0, 2.0], scale=0.7, size=(30, 2)),
                  _rng_eq.normal(loc=[-2.0, -2.0], scale=0.7, size=(30, 2))])
y_eq = np.array([1] * 30 + [-1] * 30)
_rng_test = np.random.default_rng(910)
X_test_eq = np.vstack([_rng_test.normal(loc=[2.0, 2.0], scale=0.7, size=(15, 2)),
                       _rng_test.normal(loc=[-2.0, -2.0], scale=0.7, size=(15, 2))])

_svm_eq = LinearSVCScratch(C=1.0, lr=0.01, max_iter=200, batch_size=16,
                           random_state=0).fit(X_eq, y_eq)
w_eq = [float(v) for v in _svm_eq.w_]
b_eq = float(_svm_eq.b_)
loss_final = float(_svm_eq.loss_history_[-1])
pred_test = [int(v) for v in _svm_eq.predict(X_test_eq)]
print("w =", np.round(w_eq, 6), " b =", round(b_eq, 6), " objective =", round(loss_final, 6))


In [ ]:
# Autograd's inputs: the fitted parameters, re-wrapped as leaf tensors.
_wt = torch.as_tensor(_svm_eq.w_).clone().requires_grad_(True)
_bt = torch.tensor(_svm_eq.b_, dtype=torch.float64, requires_grad=True)
_Xt = torch.as_tensor(np.asarray(X_eq, dtype=float))
_yt = torch.as_tensor(np.where(y_eq == _svm_eq.classes_[0], -1.0, 1.0))
_m = _yt * (_Xt @ _wt + _bt)
_viol = _m.detach() < 1.0
assert int(_viol.sum()) > 0, "the soft margin should leave some hinge terms active"

# Autograd reproduces the hand-written subgradient, term for term.
_loss = 0.5 * (_wt @ _wt) + _svm_eq.C * (1.0 - _m[_viol]).mean()
_loss.backward()
_hand_w = _svm_eq.w_ - _svm_eq.C * np.mean(
    _yt[_viol].numpy()[:, None] * _Xt[_viol].numpy(), axis=0)
_hand_b = -_svm_eq.C * float(_yt[_viol].mean())
assert np.max(np.abs(_wt.grad.numpy() - _hand_w)) < 1e-12, "autograd == hand formula for w"
assert abs(float(_bt.grad) - _hand_b) < 1e-12, "autograd == hand formula for b"

# At the kink z = 1 the spellings of the hinge disagree: relu picks the
# scratch's branch (grad 0), clamp takes the other (grad -1). The explicit
# `margins < 1` mask dodges the whole question.
_z1 = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
torch.relu(1.0 - _z1).backward()
_z2 = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
torch.clamp(1.0 - _z2, min=0.0).backward()
assert float(_z1.grad) == 0.0, "relu's subgradient at the kink is 0, like margins < 1"
assert float(_z2.grad) == -1.0, "clamp takes the other branch — the silent divergence"

# Training made progress on the full objective without driving hinge to zero.
_loss0 = float(_svm_eq.loss_history_[0])
assert loss_final < _loss0, "the objective should drop from epoch 0"
assert loss_final > 0.5 * float(np.dot(_svm_eq.w_, _svm_eq.w_)), "soft margin keeps hinge > 0"

# The separator separates, and only a few points carry it.
_acc = float(_svm_eq.score(X_eq, y_eq))
_sv_pts, _sv_idx = _svm_eq.support_vectors(X_eq, y_eq)
assert _acc >= 0.95, "the fixture is comfortably separable"
assert 0 < len(_sv_idx) < len(y_eq), "some, not all, points sit on or inside the margin"


### library

`LinearSVC(loss='hinge', C=C/n)` minimises the *same* strongly-convex objective — the scratch's C multiplies `mean(hinge)`, sklearn's multiplies `sum(hinge)`, hence the factor of n, and a large `intercept_scaling` keeps liblinear's bias penalty negligible. The optimiser is what differs: run to `tol=1e-8` it lands on the true minimiser (the checks show SMO agrees to ~1e-8 and no perturbation improves it), while the scratch's fixed-step loop hovers ~0.16 away in w. **What the library adds:** an exact solver — so this lane honestly exports only the test predictions, which the lanes share.

In [ ]:
import numpy as np
from sklearn.svm import SVC, LinearSVC

# hints:
# 1. The scratch's C multiplies mean(hinge); LinearSVC's C multiplies sum(hinge).
# 2. So C_lib = C / n_samples — the mean-vs-sum factor is the entire translation.
# 3. liblinear penalises b too; intercept_scaling=100 makes that penalty negligible.
# 4. The fixed-lr subgradient loop hovers far from the optimum: compare predictions.


class LinearSVCScratch:
    """sklearn's LinearSVC pointed at the scratch objective.

    The scratch minimizes 0.5*||w||^2 + C * mean(hinge); LinearSVC(loss='hinge')
    minimizes 0.5*||w||^2 + C_lib * sum(hinge), so C_lib = C / n is the exact
    translation. liblinear runs its dual coordinate descent to `tol`, so `lr`,
    `batch_size`, `max_iter` (epochs) and `random_state` have no role here —
    kept only so the constructor lines up with the scratch lanes.
    """

    def __init__(self, C=1.0, lr=0.01, max_iter=1000, batch_size=32, random_state=42):
        self.C = C
        self.lr = lr
        self.max_iter = max_iter
        self.batch_size = batch_size
        self.random_state = random_state
        self.w_ = None
        self.b_ = None
        self.classes_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n = X.shape[0]
        self._model = LinearSVC(loss='hinge', C=self.C / n, tol=1e-8,
                                max_iter=100_000, intercept_scaling=100.0).fit(X, y)
        self.w_ = self._model.coef_.ravel().astype(float)
        self.b_ = float(self._model.intercept_[0])
        return self

    def decision_function(self, X):
        """Compute w^T x + b for each sample."""
        return np.asarray(X, dtype=float) @ self.w_ + self.b_

    def predict(self, X):
        """Predict original class labels."""
        raw = (self.decision_function(X) >= 0.0).astype(int)
        return self.classes_[raw]

    def score(self, X, y):
        """Classification accuracy."""
        return np.mean(self.predict(X) == y)

    def support_vectors(self, X, y):
        """Identify support vectors: points with functional margin <= 1."""
        y_enc = np.where(y == self.classes_[0], -1.0, 1.0)
        margins = y_enc * (X @ self.w_ + self.b_)
        sv_mask = margins <= 1.0 + 1e-3  # small tolerance
        return X[sv_mask], np.where(sv_mask)[0]


In [ ]:
# exports: pred_test
_rng_eq = np.random.default_rng(909)
X_eq = np.vstack([_rng_eq.normal(loc=[2.0, 2.0], scale=0.7, size=(30, 2)),
                  _rng_eq.normal(loc=[-2.0, -2.0], scale=0.7, size=(30, 2))])
y_eq = np.array([1] * 30 + [-1] * 30)
_rng_test = np.random.default_rng(910)
X_test_eq = np.vstack([_rng_test.normal(loc=[2.0, 2.0], scale=0.7, size=(15, 2)),
                       _rng_test.normal(loc=[-2.0, -2.0], scale=0.7, size=(15, 2))])

_svm_eq = LinearSVCScratch(C=1.0).fit(X_eq, y_eq)
pred_test = [int(v) for v in _svm_eq.predict(X_test_eq)]
_obj_eq = 0.5 * _svm_eq.w_ @ _svm_eq.w_ + np.mean(
    np.maximum(0.0, 1.0 - y_eq * (X_eq @ _svm_eq.w_ + _svm_eq.b_)))
print("w =", np.round(_svm_eq.w_, 6), " b =", round(_svm_eq.b_, 6))
print("objective at the exact minimiser:", round(float(_obj_eq), 6))


In [ ]:
# Two unrelated optimisers, one objective: SMO on the dual (SVC) lands on the
# same (w, b) as liblinear's coordinate descent.
_svc = SVC(kernel='linear', C=1.0 / len(y_eq), tol=1e-10).fit(X_eq, y_eq)
assert np.max(np.abs(_svm_eq.w_ - _svc.coef_.ravel())) < 1e-6, "w agrees with SVC's dual solve"
assert abs(_svm_eq.b_ - float(_svc.intercept_[0])) < 1e-6, "b agrees with SVC's dual solve"

# It really is the minimiser: no small perturbation of (w, b) lowers the
# scratch objective (C = 1, so the mean-hinge term carries no extra factor).
_y_enc = np.where(y_eq == _svm_eq.classes_[0], -1.0, 1.0)
_o_star = 0.5 * _svm_eq.w_ @ _svm_eq.w_ + np.mean(
    np.maximum(0.0, 1.0 - _y_enc * (X_eq @ _svm_eq.w_ + _svm_eq.b_)))
_pr = np.random.default_rng(3)
_worst_gain = 0.0
for _ in range(16):
    _d = _pr.normal(size=3)
    _d /= np.linalg.norm(_d)
    _w2 = _svm_eq.w_ + 1e-3 * _d[:2]
    _b2 = _svm_eq.b_ + 1e-3 * _d[2]
    _o2 = 0.5 * _w2 @ _w2 + np.mean(np.maximum(0.0, 1.0 - _y_enc * (X_eq @ _w2 + _b2)))
    _worst_gain = max(_worst_gain, float(_o_star - _o2))
assert _worst_gain <= 1e-9, "a perturbation lowered the objective — not the minimiser"

# Less C means the margin term matters less: ||w|| shrinks monotonically.
_norms = [np.linalg.norm(LinearSVCScratch(C=c).fit(X_eq, y_eq).w_) for c in (1.0, 0.1, 0.01)]
assert _norms[0] > _norms[1] > _norms[2], "||w|| must shrink as C shrinks"

# The wrapper's sign rule matches sklearn's own predict.
_pred_ours = _svm_eq.predict(X_test_eq)
assert np.array_equal(_pred_ours, _svm_eq._model.predict(X_test_eq)), "same sign rule as sklearn"
